# $$\text{HAM10000} \approx \text{ISIC 2018}$$

In [ ]:
import json
import pandas as pd

from pathlib import Path
from collections import Counter

from datasets import load_dataset, load_from_disk, ClassLabel, DatasetDict

In [ ]:
HAM_DATASET_DOWNLOAD_PATH = Path("/home/sulcm/datasets/ham10000/ham10000_download")
HAM_DATASET_PATH = Path("/home/sulcm/datasets/ham10000/HAM10000")

## Create dataset structure

### Train

In [ ]:
ham_train_gt_one_hot = pd.read_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Training_GroundTruth.csv").set_index("image", append=True)
for column in ham_train_gt_one_hot.columns:
    ham_train_gt_one_hot[column] = ham_train_gt_one_hot[column].astype(int)
labels_train = [l.lower() for l in ham_train_gt_one_hot.columns.to_list()]
ham_train_gt_classes: pd.Series = ham_train_gt_one_hot.dot(ham_train_gt_one_hot.columns).apply(lambda x: x.lower())

In [ ]:
ham_train_gt_classes.value_counts()

In [ ]:
ham_train_data_mapping = pd.read_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Training_LesionGroupings.csv")

In [ ]:
ham_train_data_mapping

In [ ]:
(ham_train_data_mapping["image"] == ham_train_gt_classes.index.get_level_values("image")).all()

In [ ]:
ham_train_gt_classes.index = ham_train_gt_classes.index.get_level_values("image")
ham_train_data_mapping["label"] = ham_train_data_mapping["image"].map(ham_train_gt_classes)
ham_train_data_mapping["file_name"] = ham_train_data_mapping.apply(
    lambda row: row["image"] + ".jpg", axis=1
)
ham_train_data_mapping = ham_train_data_mapping.rename(columns={"image": "isic_id"})

In [ ]:
ham_train_data_mapping

In [ ]:
dset_grouped_by_label = ham_train_data_mapping.groupby("label")
dset_label_veiws = {}
for label, samples in dset_grouped_by_label.groups.items():
    dset_label_veiws[label] = ham_train_data_mapping.iloc[samples].groupby("lesion_id")

In [ ]:
# ham_train_data_mapping.to_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Training_Input" / "metadata.csv", index=False)

### Validation + Test

In [ ]:
ham_val_gt_one_hot = pd.read_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Validation_GroundTruth.csv").set_index("image", append=True)
for column in ham_val_gt_one_hot.columns:
    ham_val_gt_one_hot[column] = ham_val_gt_one_hot[column].astype(int)
labels_val = [l.lower() for l in ham_val_gt_one_hot.columns.to_list()]
ham_val_gt_classes: pd.Series = ham_val_gt_one_hot.dot(ham_val_gt_one_hot.columns).apply(lambda x: x.lower())
ham_val_data = ham_val_gt_classes.to_frame(name="label").reset_index("image")

In [ ]:
ham_val_gt_one_hot
ham_test_gt_one_hot = pd.read_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Test_GroundTruth.csv").set_index("image", append=True)
for column in ham_test_gt_one_hot.columns:
    ham_test_gt_one_hot[column] = ham_test_gt_one_hot[column].astype(int)
labels_test = [l.lower() for l in ham_test_gt_one_hot.columns.to_list()]
ham_test_gt_classes: pd.Series = ham_test_gt_one_hot.dot(ham_test_gt_one_hot.columns).apply(lambda x: x.lower())
ham_test_data = ham_test_gt_classes.to_frame(name="label").reset_index("image")

In [ ]:
ham_test_images = ham_test_data["image"]
res = [i in ham_test_images for i in ham_val_data["image"]]
any(res)

In [ ]:
ham_validation_data_mapping = pd.concat([ham_val_data, ham_test_data], ignore_index=True)
ham_validation_data_mapping = ham_validation_data_mapping.rename(columns={"image": "isic_id"})

In [ ]:
ham_validation_data_mapping[["file_name", "lesion_id"]] = ham_validation_data_mapping.apply(
    lambda row: [
        row["isic_id"] + ".jpg",
        "HAM_" + row["isic_id"]
    ],
    axis=1, result_type="expand"
)

In [ ]:
ham_validation_data_mapping

In [ ]:
# ham_validation_data_mapping.to_csv(HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Validation_Input_Concat" / "metadata.csv", index=False)

## Load/Build datatset

In [ ]:
train_dataset = load_dataset("imagefolder", data_dir=HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Training_Input")
train_dataset

In [ ]:
val_dataset = load_dataset("imagefolder", data_dir=HAM_DATASET_DOWNLOAD_PATH / "ISIC2018_Task3_Validation_Input_Concat")
val_dataset

In [ ]:
ham10000_dataset = DatasetDict({
    "train": train_dataset["train"],
    "validation": val_dataset["train"],
})
ham10000_dataset

In [ ]:
assert labels_train == labels_val == labels_test
ham10000_dataset = ham10000_dataset.cast_column("label", ClassLabel(names=labels_train))

In [ ]:
# ham10000_dataset.save_to_disk(HAM_DATASET_PATH)

## Use build HAM10000 dataset

In [ ]:
ham10000_dataset = load_from_disk(HAM_DATASET_PATH)
ham10000_dataset

In [ ]:
# ham10000_dataset.cleanup_cache_files()

## Check for intersection with MILK10k

In [ ]:
MILK10K_DATASET = Path("/home/sulcm/datasets/milk10k/milk10k")

In [ ]:
milk10k_dataset = load_from_disk(MILK10K_DATASET)
milk10k_dataset

In [ ]:
milk10k_isic_ids = list(milk10k_dataset["train"]["isic_id"])

In [ ]:
ham_train_isic_ids = list(ham10000_dataset["train"]["isic_id"])
ham_val_isic_ids = list(ham10000_dataset["validation"]["isic_id"])

In [ ]:
milk_in_ham_train = [i in ham_train_isic_ids for i in milk10k_isic_ids]
any(milk_in_ham_train)

In [ ]:
milk_in_ham_val = [i in ham_val_isic_ids for i in milk10k_isic_ids]
any(milk_in_ham_val)